# 03. Modeling

Primary model: LightGBM `LGBMRanker` with the LambdaRank objective. Variant A label encoding remaps `relevance` from {0, 1, 5} to {0, 1, 2} and uses `label_gain=[0, 1, 5]`, which calibrates the optimization to the assignment-graded NDCG.

## Hyperparameters of the deliverable

These match the anchor configuration. Thread counts are pinned to safe values; we never use `-1`.

| parameter | value |
|---|---|
| objective | lambdarank |
| metric | ndcg |
| eval_at | [5] |
| label_gain | [0, 1, 5] |
| num_leaves | 28 |
| max_depth | 9 |
| learning_rate | 0.05 |
| n_estimators | 1000 with early stopping 50 (split phase) / fixed best_iteration (deployment) |
| bagging_fraction | 0.958 |
| feature_fraction | 0.927 |
| min_child_samples | 50 |
| num_threads | 6 |
| verbose | -1 (logs only) |

## Anchor results on the locked holdout

Three seeds on the 80% / 10% / 10% split with the locked holdout. Each model is tuned with early stopping on the validation slice; the score below is on the held-out 10%.

| seed | NDCG@5 (holdout) | best_iter |
|---:|---:|---:|
| 42  | 0.393872 | 324 |
| 123 | 0.392300 | 319 |
| 456 | 0.391828 | 259 |
| **mean** | **0.392667** | |
| **std**  | **0.001070** | |

Score-averaging the three seeds produces 0.395289 on the holdout. That ensemble is the safe baseline (Kaggle public NDCG@5 = 0.39500).

## Final model: prior-augmented 3-seed LightGBM

On top of the 49-feature anchor we add the 12 leakage-safe historical prior features (block A: `prop_id`, block B: `srch_destination_id`, block C: pair). One LightGBM model per seed, fitted with the same hyperparameters but using best_iter values learned during the split phase: 291 (seed 42), 279 (seed 123), 375 (seed 456).

| seed | NDCG@5 (holdout) | best_iter |
|---:|---:|---:|
| 42  | 0.396375 | 291 |
| 123 | 0.396928 | 279 |
| 456 | 0.397011 | 375 |
| **3-seed score-avg** | **0.400682** | |

This is our final selected submission, named `submit_iter07_prior_3seed.csv` (Kaggle public NDCG@5 = 0.39690).

Feature importance from a model fitted on the same anchor plus priors (top-20 by total LightGBM gain; D-block features omitted as they belong to the rejected visitor-country prior extension):

![feature importance](../report/figures/feature_importance_lightgbm.png)

## Second technique: XGBoost `rank:ndcg`

The course rubric expects at least two ranking techniques to be compared. We compared LightGBM against XGBoost on a 5 percent stratified-by-`srch_id` sample (full-data XGBoost was prohibitively slow for our compute budget). On the sample, with identical features, XGBoost reached NDCG@5 = 0.356409 versus LightGBM 0.329601. We therefore use XGBoost as a calibration check rather than the production model: it confirms the LambdaRank class of methods dominates regression-style relevance scoring on this dataset, and the two implementations agree on the direction of feature importance.

We did not promote XGBoost to the final submission because the full-data fit on our hardware did not finish within budget and because LightGBM already provided a stronger holdout score with the leakage-safe priors.

## Ensembling explored

Two diversity directions were tested before the final submission:

1. A 5-seed extension of the anchor (seeds 42, 123, 456, 789, 2026): holdout 0.397348. With a small additive blend on per-search z-scored signals (price, star rating, location score 2, promotion flag) holdout climbed to 0.398246. Kept as a local result, not uploaded.
2. A diverse-config LightGBM (`num_leaves=63`, `learning_rate=0.035`, `lambda_l2=10`, `bagging_fraction=0.85`, `feature_fraction=0.75`) over 3 seeds, weighted 0.60 anchor / 0.40 diverse with the additive blend, reaching 0.399638 on the holdout. The diverse ensemble submission, `submit_iter06_diverse_ensemble_blend.csv`, reached Kaggle public 0.39685.

The historical-prior 3-seed model (0.400682 holdout, 0.39690 public) supersedes both. Ensembling the prior model with the diverse-config model under several weight schemes pushed the holdout up to 0.401482 at the maximum, but we did not select that blend because the optimum was identified by repeated holdout selection (see notebook 04 on shrinkage).